# 実行結果の例

```
こんにちは！
こんにちは！今日はどんなことをお手伝いできますか？

1たす2は？
1たす2は3です。何か他に質問がありますか？

台湾観光について検索結果を教えて
台湾観光に関する情報をいくつか見つけました。
1. [台湾観光局の公式サイト](https://eng.taiwan.net.tw/m1.aspx?sNo=0002005)では、国際的な競争が激化する中で台湾を世界に紹介するための取り組みが紹介されています。2023年には、台湾が世界の非OIC観光地の中で再びトップ3にランクインしました。
2. [台湾観光局の情報ページ](https://eng.taiwan.net.tw/)では、アメリカ市場での拡大やカナダでの新しい観光情報センターの設立についての最新情報が掲載されています。
台湾には美しい景色や文化がたくさんありますので、ぜひ訪れてみてください！他に知りたいことがあれば教えてください。
ありがとうございました!
```

In [5]:
# 必要なモジュールをインポート
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict
from langchain_community.tools.tavily_search import TavilySearchResults
from langgraph.graph import StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# ===== Stateクラスの定義 =====
class State(TypedDict):
    messages: Annotated[list, add_messages]

# ===== グラフの構築 =====
def build_graph(model_name):
    graph_builder = StateGraph(State)

    tool = TavilySearchResults(max_results=2)
    tools = [tool]
    
    llm = ChatOpenAI(model_name=model_name)
    llm_with_tools = llm.bind_tools(tools)
    
    def chatbot(state: State):
        return {"messages": [llm_with_tools.invoke(state["messages"])]}

    tool_node = ToolNode(tools)
    graph_builder.add_node("chatbot", chatbot)
    graph_builder.add_node("tools", tool_node)
    
    graph_builder.add_conditional_edges(
        "chatbot",
        tools_condition, # ツール呼出と判断したらツールノードを呼ぶ
    )
    graph_builder.add_edge("tools", "chatbot")
    graph_builder.set_entry_point("chatbot")
    
    memory = MemorySaver()
    return graph_builder.compile(checkpointer=memory)

# ===== グラフ実行関数 =====
def stream_graph_updates(graph: StateGraph, user_input: str):
    events = graph.stream(
        {"messages": [("user", user_input)]},
        {"configurable": {"thread_id": "1"}},
        stream_mode="values",
    )

    # 結果をストリーミングで得る
    for event in events:
        msg = event["messages"][-1]
        content = getattr(msg, "content", "")

        # human と ai だけを表示する
        if content and msg.type in ["human", "ai"]:
            print(content, flush=True)

# ===== メイン実行ロジック =====
# 環境変数の読み込み
load_dotenv("../.env")
os.environ['OPENAI_API_KEY'] = os.environ['API_KEY']

# モデル名
MODEL_NAME = "gpt-4o-mini" 

# グラフの作成
graph = build_graph(MODEL_NAME)

# メインループ
# ソースコードを記述
while True:
    user_input = input()
    if user_input.strip()=="":
        print("ありがとうございました!")
        break

    stream_graph_updates(graph, user_input)


台湾観光について検索結果を教えて
台湾観光に関する情報を以下にまとめました。

### おすすめ観光スポット

1. **台北101**
   - 高さ約509mの超高層ビルで、展望台からの360度の眺望が楽しめます。ショッピングやレストランも併設されています。

2. **国立故宮博物院**
   - 世界四大博物館のひとつで、中国の貴重な文化財が展示されています。

3. **士林夜市**
   - 台北市最大の観光夜市で、様々な台湾のローカルフードが楽しめます。

4. **九份**
   - ノスタルジックな雰囲気の町で、『千と千尋の神隠し』の舞台としても知られています。赤提灯が並ぶ風景が美しいです。

5. **日月潭**
   - 台湾最大の淡水湖で、四季折々の美しい景観が楽しめる人気スポットです。

6. **太魯閣溪谷**
   - 太魯閣国立公園内にある渓谷で、高山や滝などの自然を楽しむことができます。

7. **龍虎塔**
   - 福を得て凶を避けるため、龍の口から入り、虎の口から出るというルールがあります。

8. **地熱谷**
   - 有名な温泉地「北投温泉」の源泉で、湯気が立ち上る様子が見られます。

### 旅行のポイント
- 台湾は日本からの直行便で約4時間で行けるため、週末旅行にも最適です。
- グルメ、観光、ショッピングなど、様々な楽しみ方ができます。
- 特に夜市や地方の町を訪れることで、台湾の文化を深く体験できます。

詳細な情報は以下のリンクからも確認できます：
- [KNT台湾観光ガイド](https://www.knt.co.jp/travelguide/kaigai/027/)
- [阪急交通社台湾観光ガイド](https://www.hankyu-travel.com/guide/taiwan/) 

これらのスポットを訪れることで、台湾の魅力を存分に楽しむことができるでしょう。
ありがとうございました!
